# Breast Cancer Clinical Decision Support

**Interpretable malignancy triage for clinical review**

Regulatory framework: FDA 510(k)/De Novo SaMD pathway, IEC 62304,
ISO 13485, HIPAA, GDPR Recital 71 (automated decision explanation).

## Part 0 — Setup

In [3]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, precision_recall_fscore_support)
warnings.filterwarnings('ignore')
from hugiml import HUGIMLClassifier
from hugiml.calibration import evaluate_calibration
from hugiml.metrics import compute_all_metrics
from hugiml.pruning import PatternEditor
from hugiml.governance import generate_model_card

RANDOM_STATE = 42
DATA_FILE     = 'nb08_healthcare_breast_cancer_data.csv'
METADATA_FILE = 'nb08_healthcare_breast_cancer_metadata.csv'
TARGET = 'diagnosis_malignant'
MIN_CLINICAL_RECALL = 0.90   # Clinical safety constraint

MORPH_MAP = {
    'worst':'Worst-Case Morphology','mean radius':'Cell Size',
    'mean area':'Cell Size','mean texture':'Texture / Heterogeneity',
    'mean compact':'Compactness','mean concav':'Concavity / Irregular Contour',
    'mean symm':'Symmetry','mean fractal':'Fractal Dimension',
    'radius error':'Measurement Variability','area error':'Measurement Variability',
}

HEALTH_TEAL='#0b6e69'; HEALTH_CORAL='#d95f45'; HEALTH_AMBER='#c47d15'
import hugiml; print('hugiml-core', hugiml.__version__)


hugiml-core 1.1.2


## Part 1 — Data Loading and Privacy Controls

In [5]:
df = pd.read_csv(DATA_FILE)
metadata = pd.read_csv(METADATA_FILE)
df[TARGET] = df[TARGET].astype(int)
exclude=['patient_id','diagnosis_original_sklearn_target','diagnosis_malignant','diagnosis_label']
feature_cols=[c for c in df.columns if c not in exclude]
X=df[feature_cols]; y=df[TARGET]
print(f'Patients: {len(df)} | Features: {len(feature_cols)}')
print(f'Malignant: {int(y.sum())} ({y.mean():.2%}) | Benign: {int((1-y).sum())}')
print(f'Patient IDs excluded: ✓ (HIPAA de-identification)')
print(f'Missing values: {int(X.isna().sum().sum())}')


Patients: 569 | Features: 30
Malignant: 212 (37.26%) | Benign: 357
Patient IDs excluded: ✓ (HIPAA de-identification)
Missing values: 0


## Part 2 — Splits (60/20/20)

> **Clinical safety constraint:** For malignancy detection, recall (sensitivity) is
> the primary safety metric. A false negative (missed malignancy) carries far
> greater clinical cost than a false positive (unnecessary biopsy).
> Operating threshold is selected to achieve recall ≥ 90%.


In [7]:
clf_prep = HUGIMLClassifier(B=10, L=1, G=1e-4, topK=120)
X_enc, y_enc = clf_prep.prepareXy(X, y)
X_tr,X_tmp,y_tr,y_tmp=train_test_split(X_enc,y_enc,test_size=0.40,stratify=y_enc,random_state=RANDOM_STATE)
X_cal,X_te,y_cal,y_te=train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=RANDOM_STATE)
print(f'Train:{len(X_tr)} (mal:{y_tr.sum()})  Cal:{len(X_cal)}  Test:{len(X_te)} (mal:{y_te.sum()})')


Train:341 (mal:127)  Cal:114  Test:114 (mal:42)


## Part 3 — feature_mode Comparison

In [9]:
mode_res={}
for mode in ['patterns_only','original_plus_patterns']:
    c=HUGIMLClassifier(B=10,L=1,G=1e-4,topK=120,feature_mode=mode)
    Xe,ye=c.prepareXy(X,y)
    Xtr2,Xtmp2,ytr2,ytmp2=train_test_split(Xe,ye,test_size=0.40,stratify=ye,random_state=RANDOM_STATE)
    Xcal2,Xte2,ycal2,yte2=train_test_split(Xtmp2,ytmp2,test_size=0.50,stratify=ytmp2,random_state=RANDOM_STATE)
    c.fit(Xtr2,ytr2); p2=c.predict_proba(Xte2)[:,1]
    mode_res[mode]=dict(clf=c,Xtr=Xtr2,Xcal=Xcal2,Xte=Xte2,ytr=ytr2,ycal=ycal2,yte=yte2,proba=p2,
                        auc=roc_auc_score(yte2,p2),ap=average_precision_score(yte2,p2))
    print(f"  {mode:26s}: {len(c.get_hug_features()):3d} patterns | AUC={mode_res[mode]['auc']:.4f} | AP={mode_res[mode]['ap']:.4f}")

R=mode_res['patterns_only']
clf,X_tr,X_cal,X_te=R['clf'],R['Xtr'],R['Xcal'],R['Xte']
y_tr,y_cal,y_te,y_score=R['ytr'],R['ycal'],R['yte'],R['proba']
auc,ap=R['auc'],R['ap']
prec_c,rec_c,thr_pr=precision_recall_curve(y_te,y_score)
valid_thr=[(thr_pr[i],prec_c[i],rec_c[i]) for i in range(len(thr_pr)) if rec_c[i]>=MIN_CLINICAL_RECALL]
op_thr,prec_op0,rec_op0=max(valid_thr,key=lambda x:x[1]) if valid_thr else (0.50,0,0)
y_pred=(y_score>=op_thr).astype(int)
tn,fp,fn,tp=confusion_matrix(y_te,y_pred).ravel()
prec_op,rec_op,f1_op,_=precision_recall_fscore_support(y_te,y_pred,average='binary',zero_division=0)
print(f'Baseline @ ≥90%-recall thr {op_thr:.3f}: TP={tp}  FP={fp}  FN={fn}  recall={rec_op:.2%}  precision={prec_op:.2%}')


  patterns_only             : 120 patterns | AUC=0.9907 | AP=0.9857
  original_plus_patterns    : 120 patterns | AUC=0.9947 | AP=0.9921
Baseline @ ≥90%-recall thr 0.581: TP=39  FP=1  FN=3  recall=92.86%  precision=97.50%


## Part 4 — Calibration

In [11]:
cal_pre=evaluate_calibration(np.asarray(y_te),y_score)
print(f'ECE={cal_pre.ece:.4f}  MCE={cal_pre.mce:.4f}  Brier={cal_pre.brier_score:.4f}')


ECE=0.0537  MCE=0.4675  Brier=0.0387


## Part 5 — Interpretability Metrics

In [13]:
interp=compute_all_metrics(clf,X_te)
print(f'n_patterns:{interp.n_patterns}  coverage:{interp.coverage:.2%}  mean_active:{interp.mean_active_patterns:.2f}')
print(f'Each patient report lists ~{interp.mean_active_patterns:.0f} active morphology patterns.')


n_patterns:120  coverage:99.12%  mean_active:11.46
Each patient report lists ~11 active morphology patterns.


## Part 6 — Clinical Morphology Pattern Mapping

Patterns are annotated with clinical morphology domains:
- **Worst-case morphology**: measurements from the most abnormal cell
- **Cell size** (radius, area, perimeter): larger cells associated with malignancy
- **Texture / heterogeneity**: irregular texture indicates malignancy
- **Concavity**: irregular contours are characteristic of malignant nuclei


In [15]:
importances=clf.feature_importances().copy()
def _lbl(p):
    p=p.lower()
    for kw,l in MORPH_MAP.items():
        if kw in p: return l
    return 'Other Morphology'
importances['clinical_domain']=importances['pattern'].apply(_lbl)
importances['malignancy_signal']=np.where(importances['coefficient']>=0,'malignancy indicator','benign indicator')
top15=importances.nlargest(15,'abs_coefficient')
print('='*90)
for _,row in top15.iterrows():
    a='▲' if row['coefficient']>0 else '▼'
    print(f"  {a} {row['pattern']:46s} {row['coefficient']:+.4f}  sup={row['support']:.1%}  [{row['clinical_domain']}]")
print('='*90)
print('\nPatterns per domain:'); print(importances.groupby('clinical_domain').size().sort_values(ascending=False))


  ▲ worst symmetry=[0.3596,0.6638)                 +1.1390  sup=10.3%  [Worst-Case Morphology]
  ▲ mean texture=[21.53,23.06)                     +1.1316  sup=10.0%  [Texture / Heterogeneity]
  ▲ worst concave points=[0.1785,0.2073)           +1.0796  sup=10.0%  [Worst-Case Morphology]
  ▲ worst texture=[34.01,49.54)                    +0.9466  sup=10.3%  [Worst-Case Morphology]
  ▲ worst concave points=[0.151,0.1785)            +0.9428  sup=10.0%  [Worst-Case Morphology]
  ▲ worst area=[928.8,1269)                        +0.8359  sup=10.0%  [Worst-Case Morphology]
  ▲ worst perimeter=[117.9,133.5)                  +0.8326  sup=9.7%  [Worst-Case Morphology]
  ▲ mean concave points=[0.08293,0.09934)          +0.7974  sup=10.0%  [Concavity / Irregular Contour]
  ▼ compactness error=[0.002252,0.008974)          -0.7792  sup=10.0%  [Other Morphology]
  ▲ area error=[90.47,525.6)                       +0.7740  sup=10.3%  [Measurement Variability]
  ▼ worst radius=[12.4,13.2)                

## Part 7 — Pattern Review (IEC 62304 SaMD Governance)

SaMD governance requirements:
- Worst-group patterns retained (primary malignancy evidence)
- Low-signal noise patterns removed
- Safety check: recall must remain ≥ 90% after all changes


In [17]:
editor=PatternEditor(clf,operator_name='clinical-samd-review')
pats_df=editor.list_patterns()
print(f'Patterns before review: {len(pats_df)}')
noise_pats=pats_df[(pats_df['support']<0.05)&(pats_df['coefficient'].abs()<0.20)&(~pats_df['pattern'].str.lower().str.contains('worst'))]
print(f'Noise patterns flagged: {len(noise_pats)}')
if len(noise_pats):
    editor.remove(noise_pats['idx'].tolist(),
        reason='Low-support AND low-coefficient non-worst patterns; '
               'worst-group patterns retained as primary malignancy evidence per SaMD governance')
editor.refit(X_tr,y_tr)
editor.calibrate(X_cal,y_cal,method='isotonic')
clf_pruned=editor.finalize()

proba_pruned=clf_pruned.predict_proba(X_te)[:,1]
auc_pruned=roc_auc_score(y_te,proba_pruned)
ap_pruned=average_precision_score(y_te,proba_pruned)
cal_post=evaluate_calibration(np.asarray(y_te),proba_pruned)
prec_pp,rec_pp,thr_pp2=precision_recall_curve(y_te,proba_pruned)
valid_pp=[(thr_pp2[i],prec_pp[i],rec_pp[i]) for i in range(len(thr_pp2)) if rec_pp[i]>=MIN_CLINICAL_RECALL]
op_thr_p=max(valid_pp,key=lambda x:x[1])[0] if valid_pp else 0.50
y_pred_p=(proba_pruned>=op_thr_p).astype(int)
tn_p,fp_p,fn_p,tp_p=confusion_matrix(y_te,y_pred_p).ravel()
prec_p,rec_p,f1_p,_=precision_recall_fscore_support(y_te,y_pred_p,average='binary',zero_division=0)
print(f'After review + recalibration: AUC={auc_pruned:.4f}  AP={ap_pruned:.4f}  ECE={cal_post.ece:.4f}')
print(f'Recall={rec_p:.2%} ({"✓ MET" if rec_p>=MIN_CLINICAL_RECALL else "⚠ NOT MET"})  precision={prec_p:.2%}  FN={fn_p}')
audit_js=json.loads(editor.audit_report())
print(f'Audit: removed={audit_js["diff"]["n_removed"]}, calibrated={audit_js["calibration"]["applied"]}')


Patterns before review: 120
Noise patterns flagged: 0
After review + recalibration: AUC=0.9818  AP=0.9550  ECE=0.0369
Recall=90.48% (✓ MET)  precision=92.68%  FN=4
Audit: removed=0, calibrated=True


## Part 8 — Clinical Threshold Analysis (Recall-First)

False negatives (missed malignancies) carry greater clinical cost than false positives.
We present three operating points by minimum recall target.


In [19]:
thresh_rows=[]
for thr in np.linspace(0.01,0.99,49):
    pred=(proba_pruned>=thr).astype(int)
    if pred.sum()==0: continue
    tn_t,fp_t,fn_t,tp_t=confusion_matrix(np.asarray(y_te),pred,labels=[0,1]).ravel()
    prec_t,rec_t,f1_t,_=precision_recall_fscore_support(y_te,pred,average='binary',zero_division=0)
    thresh_rows.append({'threshold':thr,'tp':tp_t,'fp':fp_t,'fn':fn_t,
        'precision':prec_t,'recall':rec_t,'f1':f1_t,'specificity':tn_t/max(tn_t+fp_t,1)})
threshold_table=pd.DataFrame(thresh_rows)
print('Clinical operating points:')
for tr in [0.90,0.95,0.99]:
    r=threshold_table[threshold_table['recall']>=tr]
    if len(r):
        row=r.iloc[-1]
        print(f"  {tr:.0%} recall: thr={row.threshold:.3f}  precision={row.precision:.2%}  FN={int(row.fn)}  specificity={row.specificity:.2%}")
threshold_table.round(4)


Clinical operating points:
  90% recall: thr=0.561  precision=92.68%  FN=4  specificity=95.83%
  95% recall: thr=0.275  precision=85.11%  FN=2  specificity=90.28%
  99% recall: thr=0.133  precision=70.00%  FN=0  specificity=75.00%


,threshold,tp,fp,fn,precision,recall,f1,specificity
0,0.0100,42,21,0,0.6667,1.0000,0.8000,0.7083
1,0.0304,42,21,0,0.6667,1.0000,0.8000,0.7083
2,0.0508,42,21,0,0.6667,1.0000,0.8000,0.7083
3,0.0712,42,21,0,0.6667,1.0000,0.8000,0.7083
4,0.0917,42,19,0,0.6885,1.0000,0.8155,0.7361
5,0.1121,42,18,0,0.7000,1.0000,0.8235,0.7500
6,0.1325,42,18,0,0.7000,1.0000,0.8235,0.7500
7,0.1529,41,7,1,0.8542,0.9762,0.9111,0.9028
8,0.1733,41,7,1,0.8542,0.9762,0.9111,0.9028
9,0.1938,40,7,2,0.8511,0.9524,0.8989,0.9028


## Part 9 — Morphology Strata Safety Review

In [21]:
test_idx=X_te.index if hasattr(X_te,'index') else pd.Index(range(len(y_te)))
raw_test=X.loc[test_idx].copy()
af=raw_test.copy(); af['_actual']=np.asarray(y_te).astype(int)
af['_score']=proba_pruned; af['_flag']=y_pred_p
strata_rows=[]
for col in ['mean compactness','worst area','worst concave points']:
    if col not in af.columns: continue
    try:
        af[f'_s_{col}']=pd.qcut(af[col],3,labels=['Low','Mid','High'],duplicates='drop')
        for level in ['Low','Mid','High']:
            g=af[af[f'_s_{col}'].eq(level)]
            if len(g)<15: continue
            yg=g['_actual'].to_numpy(); fg=g['_flag'].to_numpy()
            auc_g=roc_auc_score(yg,g['_score']) if len(np.unique(yg))==2 else np.nan
            tn_g,fp_g,fn_g,tp_g=confusion_matrix(yg,fg,labels=[0,1]).ravel()
            strata_rows.append({'feature':col,'stratum':level,'n':len(g),
                'malignant_rate':yg.mean(),'flag_rate':fg.mean(),'auc':auc_g,
                'recall':tp_g/max(tp_g+fn_g,1),'precision':tp_g/max(tp_g+fp_g,1)})
    except Exception: pass
strata_audit=pd.DataFrame(strata_rows)
print(strata_audit.sort_values(['feature','stratum']).round(4))
print('\nSafety check: recall ≥ 90% in each stratum?')
for _,row in strata_audit.iterrows():
    flag='✓' if row['recall']>=MIN_CLINICAL_RECALL or pd.isna(row['recall']) else '⚠ BELOW TARGET'
    print(f"  {row['feature']} [{row['stratum']}]: recall={row['recall']:.2%}  {flag}")


                feature stratum   n  malignant_rate  flag_rate     auc  \
2      mean compactness    High  38          0.7632     0.8158  0.9330   
0      mean compactness     Low  38          0.0789     0.0789  0.9905   
1      mean compactness     Mid  38          0.2632     0.1842  0.9679   
5            worst area    High  38          0.9211     0.8947  0.9714   
3            worst area     Low  38          0.0000     0.0263     NaN   
4            worst area     Mid  38          0.1842     0.1579  0.9378   
8  worst concave points    High  38          0.8947     0.8947  0.9632   
6  worst concave points     Low  38          0.0000     0.0000     NaN   
7  worst concave points     Mid  38          0.2105     0.1842  0.9333   

   recall  precision  
2  1.0000     0.9355  
0  0.6667     0.6667  
1  0.7000     1.0000  
5  0.9429     0.9706  
3  0.0000     0.0000  
4  0.7143     0.8333  
8  0.9706     0.9706  
6  0.0000     0.0000  
7  0.6250     0.7143  

Safety check: recall ≥ 90% i

## Part 10 — Covariate Drift (PSI)

In [23]:
def compute_psi(expected,actual,buckets=10):
    rows=[]
    common=expected.select_dtypes(include=[np.number]).columns.intersection(actual.columns)
    for col in common:
        edges=np.percentile(expected[col].dropna(),np.linspace(0,100,buckets+1))
        edges[0]=-np.inf; edges[-1]=np.inf
        ep=np.maximum(np.histogram(expected[col],bins=edges)[0]/len(expected),1e-6)
        ap_=np.maximum(np.histogram(actual[col],bins=edges)[0]/len(actual),1e-6)
        psi=float(np.sum((ap_-ep)*np.log(ap_/ep)))
        rows.append({'feature':col,'psi':round(psi,4),
            'status':'STABLE' if psi<0.10 else 'WARNING' if psi<0.25 else 'SHIFT'})
    return pd.DataFrame(rows).sort_values('psi',ascending=False)

rng_d=np.random.default_rng(77); n_d=100
X_num=X.select_dtypes(include=[np.number])
X_raw_train=X_num.iloc[:340]
# Simulate scanner calibration shift
X_raw_drift=X_raw_train.sample(n=n_d,replace=True,random_state=77).copy()
X_raw_drift=X_raw_drift*rng_d.uniform(1.05,1.25,X_raw_drift.shape)
X_raw_drift.index=range(n_d)
psi_df=compute_psi(X_raw_train,X_raw_drift)
print('PSI vs simulated scanner protocol shift:'); print(psi_df.head(10).to_string(index=False))


PSI vs simulated scanner protocol shift:
                feature    psi status
 mean fractal dimension 3.3313  SHIFT
       worst smoothness 1.8718  SHIFT
          mean symmetry 1.6999  SHIFT
        mean smoothness 0.8554  SHIFT
worst fractal dimension 0.6691  SHIFT
         worst symmetry 0.5750  SHIFT
           mean texture 0.5457  SHIFT
            mean radius 0.5247  SHIFT
           worst radius 0.5106  SHIFT
        worst perimeter 0.4621  SHIFT


## Part 11 — Model Governance (SaMD / IEC 62304)

In [25]:
card=generate_model_card(
    clf_pruned, model_id='breast-cancer-cds-v1.0',
    intended_use='Clinical decision support for breast cancer malignancy triage. Decision support only.',
    out_of_scope_use='NOT a diagnostic device. Not for autonomous diagnosis or treatment decisions.',
    training_data_description=f'UCI Breast Cancer Wisconsin Dataset, {len(df)} patients, 60/20/20 split.',
    evaluation_data_description=f'Stratified 20% holdout; {int(y_te.sum())} malignant cases.',
    performance_metrics={'AUC-ROC':round(auc_pruned,4),'AvgPrecision':round(ap_pruned,4),
        'Recall_clinical':round(float(rec_p),4),'Precision':round(float(prec_p),4),
        'ECE':round(cal_post.ece,4),'FalseNegatives':fn_p},
    limitations=[
        'Small dataset (569 patients) — external validation required before clinical deployment.',
        'FNA data only — not validated for other biopsy or imaging modalities.',
        'No demographic attributes — population representativeness cannot be verified.',
    ],
    ethical_considerations='All outputs require clinician review. Pattern explanations mandatory per GDPR Rec.71. IEC 62304 change control required for all updates.'
)
print(card.to_markdown())
card.save('nb08_healthcare_breast_cancer_model_card.json')
editor.save_audit_report('nb08_healthcare_breast_cancer_audit_trail.json')
importances.to_csv('nb08_healthcare_breast_cancer_pattern_inventory.csv',index=False)
threshold_table.to_csv('nb08_healthcare_breast_cancer_threshold_grid.csv',index=False)
strata_audit.to_csv('nb08_healthcare_breast_cancer_strata_audit.csv',index=False)
print('\n✓ Governance artifacts saved.')


# Model Card: breast-cancer-cds-v1.0

**Type:** HUGIMLClassifierNative  
**License:** Apache-2.0  
**Created:** 2026-05-29T06:37:01Z  
**Framework:** hugiml-core 1.1.2

## Reference

Krishnamoorthy, S. (2024). Interpretable Classifier Models for Decision Support Using High Utility Gain Patterns. IEEE Access, 12, 126088-126107. DOI: 10.1109/ACCESS.2024.3455563

## Intended Use

Clinical decision support for breast cancer malignancy triage. Decision support only.

## Out-of-Scope Use

NOT a diagnostic device. Not for autonomous diagnosis or treatment decisions.

## Training Data

UCI Breast Cancer Wisconsin Dataset, 569 patients, 60/20/20 split.

## Evaluation Data

Stratified 20% holdout; 42 malignant cases.

## Hyperparameters

- **B**: 10
- **L**: 1
- **G**: 0.0001
- **topK**: 120
- **adaptive_binning**: False
- **feature_mode**: patterns_only

## Performance Metrics

- **AUC-ROC**: 0.9818
- **AvgPrecision**: 0.955
- **Recall_clinical**: 0.9048
- **Precision**: 0.9268
- **ECE**: 0.036